# Chapter 20: Graph Engineering — Wiring the GAS Agents into a LangGraph

## 🎯 Chapter Goal
Build the GAS (Guardian Angel System) multi-agent graph using LangGraph primitives: typed state with Annotated reducers, conditional routing, fan-out parallelism, and MemorySaver checkpointing.

## 📖 Theory Recap

### Why Graphs Over Loops
Traditional agent loops are implicit: a `while` loop with `if/else` routing buried inside. LangGraph makes the topology **explicit and inspectable**:
- **Nodes** = agent functions (pure `state → state` transforms)
- **Edges** = deterministic transitions between nodes
- **Conditional edges** = LLM-driven or rule-driven routing
- **Typed state** = shared memory with reducer semantics

### Graph Primitives

| Primitive | Purpose |
|-----------|----------|
| `StateGraph` | Container for nodes + edges |
| `START` / `END` | Entry and exit sentinels |
| `add_node` | Register an agent function |
| `add_edge` | Deterministic transition |
| `add_conditional_edges` | Routing based on state value |
| `Send` API | Fan-out to parallel branches |

### Typed State with Annotated Reducers
```python
from typing import Annotated
import operator

class GASState(TypedDict):
    agent_messages: Annotated[list, operator.add]  # append-only reducer
    alert_level: str                               # last-write-wins
```
The `Annotated[list, operator.add]` reducer means each node's returned list is **appended** to the existing list, not replaced.

### Fan-out / Fan-in Parallelism
The `Send` API lets a supervisor node spawn multiple parallel branches:
```python
from langgraph.types import Send

def supervisor_node(state):
    if state["alert_level"] == "critical":
        return [Send("advisor_node", state), Send("safety_node", state)]
```

### Checkpointing
`MemorySaver` persists the full state after every node execution — enabling:
- **Human-in-the-loop**: pause and resume at any node
- **Fault tolerance**: resume from last checkpoint after failure
- **State inspection**: query historical state at any step

### Subgraphs
Each GAS agent can itself be a `StateGraph`, enabling modular composition and independent testing of each agent subsystem.

## 🔨 Build It

In [ ]:
# Install dependencies
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "langgraph", "langchain-openai"],
    check=False
)
print("✅ Dependencies installed")

In [ ]:
import os
import operator
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# ── Test-mode flag ──────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))
from llm_utils import llm, TEST_MODE, BACKEND, MODEL

# ── GAS State ───────────────────────────────────────────────────────────────
class GASState(TypedDict):
    patient_id: str
    glucose_value: float
    alert_level: str           # none | low | medium | high | critical
    recommendation: str
    safety_approved: bool
    agent_messages: Annotated[list, operator.add]  # append-only reducer
    next_agent: str

print("✅ GASState defined")
print("Fields:", list(GASState.__annotations__.keys()))
print("\nAnnotated reducer on agent_messages: each node's list is APPENDED, not replaced")

In [ ]:
# ── Agent Node Functions ─────────────────────────────────────────────────────

def monitor_node(state: GASState) -> dict:
    """Monitor Agent: reads glucose value and classifies alert level."""
    glucose = state["glucose_value"]

    if glucose < 54:
        alert = "critical"
    elif glucose < 70:
        alert = "high"
    elif glucose < 80:
        alert = "medium"
    elif glucose > 250:
        alert = "critical"
    elif glucose > 180:
        alert = "high"
    elif glucose > 140:
        alert = "medium"
    else:
        alert = "none"

    msg = f"[Monitor] Glucose={glucose} mg/dL → alert_level={alert}"
    print(msg)
    return {
        "alert_level": alert,
        "agent_messages": [msg],
        "next_agent": "supervisor"
    }


def advisor_node(state: GASState) -> dict:
    """Advisor Agent: generates a clinical recommendation."""
    glucose = state["glucose_value"]
    alert = state["alert_level"]

    if TEST_MODE or BACKEND != "openai":
        # Deterministic mock recommendations
        if alert == "critical" and glucose < 70:
            rec = "URGENT: Consume 15g fast-acting carbs immediately. Recheck in 15 min."
        elif alert in ("high", "critical") and glucose > 180:
            rec = "Consider correction dose. Avoid high-carb foods. Increase hydration."
        elif alert == "medium":
            rec = "Monitor closely. Light snack if trending down. Recheck in 30 min."
        else:
            rec = "Glucose within target range. Continue current management plan."
    else:
        # llm_utils handles backend selection automatically
        prompt = f"Patient glucose: {glucose} mg/dL, alert: {alert}. Give a 1-sentence clinical recommendation."
        rec = llm.chat([{"role": "user", "content": prompt}])

    msg = f"[Advisor] Recommendation: {rec}"
    print(msg)
    return {"recommendation": rec, "agent_messages": [msg]}


def safety_node(state: GASState) -> dict:
    """Safety Critic: validates the recommendation for safety."""
    rec = state.get("recommendation", "")
    alert = state["alert_level"]

    # Rule-based safety checks
    dangerous_keywords = ["ignore", "skip insulin", "no treatment needed"]
    is_safe = not any(kw in rec.lower() for kw in dangerous_keywords)

    # Critical glucose must have urgent language
    if alert == "critical":
        is_safe = is_safe and any(
            kw in rec.lower() for kw in ["urgent", "immediately", "correction", "now"]
        )

    msg = f"[Safety] Recommendation {'✅ APPROVED' if is_safe else '❌ REJECTED'}"
    print(msg)
    return {"safety_approved": is_safe, "agent_messages": [msg]}


def caregiver_node(state: GASState) -> dict:
    """Caregiver Notifier: sends notification for high/critical alerts."""
    alert = state["alert_level"]
    patient_id = state["patient_id"]

    if alert in ("high", "critical"):
        msg = (
            f"[Caregiver] 🚨 ALERT sent for patient {patient_id}: "
            f"{alert.upper()} — {state.get('recommendation', '')[:80]}"
        )
    else:
        msg = f"[Caregiver] No notification needed (alert_level={alert})"

    print(msg)
    return {"agent_messages": [msg]}


def supervisor_node(state: GASState) -> dict:
    """Supervisor Agent: orchestrates routing decisions."""
    msg = f"[Supervisor] Routing — alert_level={state['alert_level']}"
    print(msg)
    return {"agent_messages": [msg]}


print("✅ All 5 agent node functions defined")

In [ ]:
# ── Routing Functions ────────────────────────────────────────────────────────

def route_after_monitor(state: GASState) -> Literal["supervisor_node"]:
    """After monitoring, always route to supervisor."""
    return "supervisor_node"


def route_from_supervisor(state: GASState) -> Literal["advisor_node", "__end__"]:
    """Supervisor decides: escalate to advisor or exit."""
    alert = state["alert_level"]
    if alert in ("critical", "high", "medium"):
        return "advisor_node"
    else:
        return END  # Normal glucose — no further action


def route_after_safety(state: GASState) -> Literal["caregiver_node", "advisor_node"]:
    """After safety check: approved → caregiver, rejected → re-advise."""
    if state.get("safety_approved", False):
        return "caregiver_node"
    else:
        return "advisor_node"  # Loop back for revision


print("✅ Routing functions defined")
print("  route_after_monitor  → always supervisor_node")
print("  route_from_supervisor → advisor_node (alert) | END (normal)")
print("  route_after_safety   → caregiver_node (approved) | advisor_node (rejected)")

In [ ]:
# ── Build the LangGraph ──────────────────────────────────────────────────────

checkpointer = MemorySaver()
builder = StateGraph(GASState)

# Register nodes
builder.add_node("monitor_node",    monitor_node)
builder.add_node("supervisor_node", supervisor_node)
builder.add_node("advisor_node",    advisor_node)
builder.add_node("safety_node",     safety_node)
builder.add_node("caregiver_node",  caregiver_node)

# Entry point: START → monitor
builder.add_edge(START, "monitor_node")

# monitor → supervisor (conditional, always routes to supervisor)
builder.add_conditional_edges("monitor_node", route_after_monitor)

# supervisor → advisor (if alert) or END (if normal)
builder.add_conditional_edges("supervisor_node", route_from_supervisor)

# advisor → safety (deterministic)
builder.add_edge("advisor_node", "safety_node")

# safety → caregiver (approved) or advisor (rejected — feedback loop)
builder.add_conditional_edges("safety_node", route_after_safety)

# caregiver → END
builder.add_edge("caregiver_node", END)

# Compile with MemorySaver checkpointing
gas_graph = builder.compile(checkpointer=checkpointer)

print("✅ GAS LangGraph compiled successfully")
print(f"   Nodes: {list(builder.nodes.keys())}")
print(f"   Checkpointer: {type(checkpointer).__name__}")

In [ ]:
# ── Visualize the Graph ──────────────────────────────────────────────────────
try:
    from IPython.display import Image, display
    img_data = gas_graph.get_graph().draw_mermaid_png()
    display(Image(img_data))
    print("✅ Graph visualization rendered")
except Exception as e:
    # Fallback: print Mermaid source
    try:
        mermaid_src = gas_graph.get_graph().draw_mermaid()
        print("Mermaid diagram source:")
        print(mermaid_src)
    except Exception:
        print(f"Visualization unavailable: {e}")
        print("\nGraph topology (text representation):")
        print("  START")
        print("    └─► monitor_node")
        print("          └─► supervisor_node")
        print("                ├─► advisor_node  (if alert in critical/high/medium)")
        print("                │     └─► safety_node")
        print("                │           ├─► caregiver_node  (if approved)")
        print("                │           │     └─► END")
        print("                │           └─► advisor_node    (if rejected — loop)")
        print("                └─► END  (if alert == none)")

In [ ]:
# ── Run 3 Clinical Scenarios ─────────────────────────────────────────────────

scenarios = [
    {"name": "Normal Glucose",  "patient_id": "P001", "glucose_value": 105.0},
    {"name": "Hypoglycemia",    "patient_id": "P002", "glucose_value": 58.0},
    {"name": "Hyperglycemia",   "patient_id": "P003", "glucose_value": 285.0},
]

results = []

for i, scenario in enumerate(scenarios):
    print(f"\n{'='*60}")
    print(f"🧪 Scenario {i+1}: {scenario['name']} (glucose={scenario['glucose_value']} mg/dL)")
    print('='*60)

    initial_state: GASState = {
        "patient_id":      scenario["patient_id"],
        "glucose_value":   scenario["glucose_value"],
        "alert_level":     "none",
        "recommendation":  "",
        "safety_approved": False,
        "agent_messages":  [],
        "next_agent":      "monitor",
    }

    config = {"configurable": {"thread_id": f"thread-{i}"}}
    final_state = gas_graph.invoke(initial_state, config=config)

    print(f"\n📊 Final State:")
    print(f"   alert_level    : {final_state['alert_level']}")
    print(f"   recommendation : {final_state.get('recommendation', 'N/A')[:80]}")
    print(f"   safety_approved: {final_state.get('safety_approved', False)}")
    print(f"   messages       : {len(final_state['agent_messages'])} messages")

    results.append(final_state)

print(f"\n✅ All 3 scenarios completed")

In [ ]:
# ── Checkpoint Inspection ────────────────────────────────────────────────────
print("🔍 Inspecting checkpointed state for thread-1 (Hypoglycemia):")

config = {"configurable": {"thread_id": "thread-1"}}
checkpoint = gas_graph.get_state(config)

print(f"   Values keys    : {list(checkpoint.values.keys())}")
print(f"   Alert level    : {checkpoint.values.get('alert_level')}")
print(f"   Messages count : {len(checkpoint.values.get('agent_messages', []))}")
print(f"   Next steps     : {checkpoint.next}")
print(f"   Config         : {checkpoint.config}")

print("\n✅ Checkpointing works — full state persisted across invocations")
print("   This enables human-in-the-loop: pause at any node, inspect, resume")

In [ ]:
# ── Message Trace ────────────────────────────────────────────────────────────
print("📋 Full agent message trace for Hypoglycemia scenario:")
hypo_state = results[1]  # Hypoglycemia result
for i, msg in enumerate(hypo_state["agent_messages"], 1):
    print(f"  {i:2d}. {msg}")

print(f"\n📊 Summary across all 3 scenarios:")
print(f"{'Scenario':<20} {'Glucose':>8} {'Alert':>10} {'Approved':>10} {'Messages':>10}")
print("-" * 62)
for s, r in zip(scenarios, results):
    print(
        f"{s['name']:<20} {s['glucose_value']:>8.1f} "
        f"{r['alert_level']:>10} {str(r.get('safety_approved', False)):>10} "
        f"{len(r['agent_messages']):>10}"
    )

## 🏥 GAS Connection

This notebook **IS** the GAS system architecture. The `StateGraph` we built maps directly to the 5-agent GAS design:

| LangGraph Concept | GAS Component | Role |
|-------------------|---------------|------|
| `GASState` TypedDict | Shared patient context | Single source of truth |
| `monitor_node` | Monitor Agent | CGM reader + anomaly detection |
| `supervisor_node` | Supervisor Agent | Orchestrator + router |
| `advisor_node` | Advisor Agent | RAG + memory recommendations |
| `safety_node` | Safety Critic | Reward-model validation |
| `caregiver_node` | Caregiver Notifier | Human escalation via A2A |
| Conditional edges | Alert-level routing | Escalation protocol |
| `MemorySaver` | Fault-tolerant persistence | Checkpoint + resume |

**Key design decisions:**
1. `Annotated[list, operator.add]` on `agent_messages` creates an **audit trail** — every agent appends its message, nothing is lost
2. The safety feedback loop (`safety_node → advisor_node`) implements **self-correction** without external intervention
3. `MemorySaver` enables **human-in-the-loop**: a caregiver can inspect state mid-execution and override the recommendation

## ✅ Checkpoint

**What we built:**
- ✅ `GASState` TypedDict with `Annotated[list, operator.add]` reducer for message accumulation
- ✅ 5 agent node functions: monitor, supervisor, advisor, safety, caregiver
- ✅ Conditional routing: alert-level-based escalation protocol
- ✅ Safety feedback loop: rejected recommendations loop back to advisor
- ✅ `MemorySaver` checkpointing: full state persisted after every node
- ✅ Tested on 3 clinical scenarios: normal (exits early), hypoglycemia (full pipeline), hyperglycemia (full pipeline)
- ✅ Message trace: complete audit trail of all agent actions

**Key insight:** LangGraph's explicit graph topology makes the GAS coordination protocol **inspectable, testable, and modifiable** — unlike implicit while-loops where routing logic is buried in code.

**Next:** Chapter 21 — Agent Design Patterns (ReAct, Reflection, Plan-and-Execute applied to GAS)